# ColdSite-DTI cloud harness -- CANARY -- does the harness reproduce a committed cell?

Generated by `notebooks/build_harness_notebook.py` from `config/canary_wave.json`; do not hand-edit.

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (the pre-flight refuses anything else) |
| Internet | **On** |
| Environment | **Pin to original** |
| How to run | **Save Version -> Save & Run All (Commit)** |
| First run | leave `DRY_RUN = True`: it does the pre-flight and prints the plan, then stops |

## What this account trains (1 cell(s))

| GPU queue | cell |
|---|---|
| gpu0 | davis_cold_target_coldsite_dti_seed1 |

Every cell of the plan appears once in `config/canary_wave.json` across all accounts. Nothing here trains a
cell another account owns.

## The loop (each commit starts empty)

Kaggle output is per version. After a commit ends, download `results/` (the runner zips it, section 6),
add it as a private dataset, set `RESTORE_FROM = '/kaggle/input'` and run again. The runner copies back only
this account's cells, never over a file already there, verifies each finished cell's marker, and continues
partial cells from their last finished epoch. A commit self-stops before the session limit
(`config/harness.json`).

## How long this takes (Kaggle 2 x T4)

| part | approx. wall-clock | where the number comes from |
|---|---|---|
| setup: clone, data, splits | 5-10 min | the Kaggle logs of earlier runs |
| the canary cell (one GPU; the other waits) | **about 1.0 to 1.3 h** (27 to 38 epochs) | the two recorded attempts: 3,529 s and 4,727 s (`docs/canary_attempts/README.md`) |
| the epoch-1 gate (3 one-epoch runs on the two GPUs) | about 5 to 10 min | one recorded gate run |
| **whole notebook** | **about 1.3 to 1.7 h** | |

It spends about 1 to 1.5 GPU-hours of the account's weekly quota (the quota line in the monitor shows it live).

## What you will see while it runs, and what is saved

**Time checker.** Every 5 minutes the runner prints a block like

```
[monitor] elapsed 2h14m | self-stop in 8h01m (session limit 11 h) | cells complete 2/12 | quota used 4.6 of 25.5 gpu-hours
  gpu0: davis_random_hyperattentiondti_seed4 epoch 7, 361.2 s/epoch | this cell finishes within 3h20m (upper bound; ...) | 5 cell(s) left in its queue
  gpu1: ...
```

`self-stop in` is the time left before the harness stops cleanly (the session limit minus a 45-minute margin); a time estimate for a cell is an
UPPER bound because early stopping can end it sooner. Sections 2 to 4 also print the time since the session began.

**Progress saver.** Nothing is saved only at the end:
* every epoch each cell writes its resume file (atomically) and a status file, so a stop at any moment loses at most one epoch;
* a cell is marked complete only when its checkpoint, results and predictions hash are written and agree;
* every 5 minutes `results/progress.json` is rewritten (time used, cells complete, what each GPU is on);
* section 6 zips `results/` (including `progress.json` and the quota ledger) into the file you download.
If the session dies, download whatever Kaggle kept, add it as a dataset and run again with `RESTORE_FROM = '/kaggle/input'`.

## 1. Settings

In [ ]:
ACCOUNT = 'CANARY'
WAVES = 'config/canary_wave.json'
COMMIT = 'cf15813a4738c38ab05fcfdeaeb27c95cf75affb'        # the exact commit this notebook was generated for
REPO = 'https://github.com/Mahim56207/ColdSite-DTI_New.git'
DRY_RUN = True            # True: pre-flight + plan, then stop. False: train (spends GPU quota).
RESTORE_FROM = None       # None on the first commit; '/kaggle/input' after attaching the previous output

## 2. Session start and GPUs

In [ ]:
import os, subprocess, sys, time
START = time.time()          # the runner's self-stop is measured from here
def since_start():
    m = int((time.time() - START) // 60)
    return f'[time] {m // 60}h{m % 60:02d}m since the session began'

print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'nvidia-smi: none')

## 3. Clone the repo at the pinned commit

In [ ]:
WORK = '/kaggle/working'
SRC = f'{WORK}/ColdSite-DTI_New'
if not os.path.exists(SRC):
    subprocess.run(['git', 'clone', REPO, SRC], check=True)
os.chdir(SRC)
subprocess.run(['git', 'fetch', '--all', '-q'], check=True)
subprocess.run(['git', 'checkout', '-q', COMMIT], check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert head == COMMIT, f'checked out {head}, expected {COMMIT}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tabulate', 'subword-nmt'], check=True)
print('repo at', head)
print(since_start())

## 4. Raw data and splits

The same steps every earlier notebook ran. The runner's pre-flight then checks the result against
`data/splits/MANIFEST.json` and `config/cloud_manifest.json` (SHA-256), which is stricter than the row counts
the older notebooks compared.

In [ ]:
BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data'
for ds in ('davis', 'kiba'):
    os.makedirs(f'src/data/baselines/deepdta/data/{ds}', exist_ok=True)
    for fname in ('ligands_can.txt', 'proteins.txt', 'Y'):
        target = f'src/data/baselines/deepdta/data/{ds}/{fname}'
        if not os.path.exists(target):
            subprocess.run(['curl', '-sL', f'{BASE}/{ds}/{fname}', '-o', target], check=True)
subprocess.run([sys.executable, '-m', 'src.data.load_data'], check=True)
subprocess.run([sys.executable, '-m', 'src.data.build_splits'], check=True)
print(since_start())

## 5. Run: pre-flight, restore, one process per GPU

In [ ]:
RESULTS = f'{WORK}/results'
cmd = [sys.executable, '-u', '-m', 'src.cloud.runner', '--account', ACCOUNT, '--waves', WAVES,
       '--results-root', RESULTS, '--session-start', str(START)]
if RESTORE_FROM:
    cmd += ['--restore-from', RESTORE_FROM]
if DRY_RUN:
    cmd += ['--dry-run']
print(' '.join(cmd))
code_ = subprocess.run(cmd).returncode
print('runner exit code', code_, '(0 ok, 1 a cell needs attention, 2 refused by the pre-flight)')

## 5b. The verdict

Pass rule: `docs/PROTOCOL_AMENDMENT_v2.md` section 6 with the tolerance declared before this re-run: both metrics within **4** committed-seed SDs, same seed (`src/cloud/canary.py`, rule history in its docstring).

In [ ]:
if not DRY_RUN:
    verdict = subprocess.run([sys.executable, '-m', 'src.cloud.canary', '--new', RESULTS,
                              '--out', f'{RESULTS}/canary_verdict.json'])
    print('canary exit code', verdict.returncode, '(0 pass, 1 fail, 3 inconclusive: go back to the user)')
else:
    print('DRY_RUN: no canary was trained.')

## 5c. The epoch-1 gate

One epoch each of the canary cell at the committed seed (1, the replay) and at the new wave seeds (4 and 5). Two verdicts are written from
the SAME three runs: `epoch_gate_verdict_original_rule.json` (the rule as first written; it can fail a healthy harness) and
**`epoch_gate_verdict.json`, the official one**, from `src/cloud/gate_rule_v2.py` (Gate B is the family-wise prediction interval of a new draw; Gate A,
finiteness and distinctness unchanged; `docs/canary_attempts/README.md` says why). About 5 to 10 minutes on the two T4s.

In [ ]:
if not DRY_RUN:
    gate = subprocess.run([sys.executable, '-m', 'src.cloud.epoch_gate', '--run', '--seeds', '1', '4', '5',
                           '--root', f'{RESULTS}/epoch_gate', '--out', f'{RESULTS}/epoch_gate_verdict_original_rule.json'])
    print('original-rule exit code', gate.returncode, '(0 pass, 1 fail, 3 inconclusive); the runs are on disk either way')
    official = subprocess.run([sys.executable, '-m', 'src.cloud.gate_rule_v2', '--evaluate', '--seeds', '1', '4', '5',
                               '--root', f'{RESULTS}/epoch_gate', '--out', f'{RESULTS}/epoch_gate_verdict.json'])
    print('OFFICIAL epoch gate exit code', official.returncode, '(0 pass, 1 fail, 3 inconclusive: go back to the user)')
    used_h = (time.time() - START) / 3600
    print(f'This account has now spent up to {2 * used_h:.2f} GPU-hours (2 GPUs x {used_h:.2f} h) outside the wave. Record it:')
    print(f"  python -m src.cloud.quota --root results --account <ACCOUNT> --add-external {2 * used_h:.2f} "
          f"--external-id canary-{time.strftime('%Y-%m-%d')} --note 'canary + epoch gate'")
    print(since_start())
else:
    print('DRY_RUN: the epoch gate was not run.')

## 6. What landed, and the file to take with you

In [ ]:
import glob, json
for path in sorted(glob.glob(f'{RESULTS}/runner_summary_*.json')):
    for row in json.load(open(path))['cells']:
        print(f"{row['outcome']:24s} {row['cell']}  {row['detail']}")
print()
print('complete-markers:', len(glob.glob(f'{RESULTS}/**/*_complete.json', recursive=True)))
prog = f'{RESULTS}/progress.json'
if os.path.exists(prog):
    pr = json.load(open(prog))
    print(f"progress.json: {pr['cells_complete']} of {pr['cells_total']} cells complete; session {pr['elapsed_s'] / 3600:.2f} h long")
    if pr['cells_complete'] < pr['cells_total'] and not DRY_RUN:
        print("NOT FINISHED: download the zip below, add it as a private dataset, set RESTORE_FROM = '/kaggle/input' and run this notebook again (the next session continues each cell from its last epoch).")
    elif pr['cells_total'] and not DRY_RUN:
        print('ALL CELLS COMPLETE for this account: keep the zip; analysis needs it.')
print(since_start())
ZIP = f'{ACCOUNT.lower()}_results.zip'
subprocess.run(f'cd {WORK} && rm -f {ZIP} && zip -qr {ZIP} results', shell=True)
print(f'{WORK}/{ZIP} -- download from the Output panel, add it to your restore dataset')